In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
!apt-get update
!apt-get install -y nvidia-cuda-toolkit

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntu

In [3]:
from google.colab import files
uploaded = files.upload()

Saving ex01_cuda_basics.cu to ex01_cuda_basics.cu
Saving ex02_memory_hierarchy.cu to ex02_memory_hierarchy.cu
Saving ex03_ml_primitives.cu to ex03_ml_primitives.cu
Saving ex04_cnn_layers.cu to ex04_cnn_layers.cu
Saving ex05_mnist_cnn.cu to ex05_mnist_cnn.cu
Saving Makefile to Makefile


In [4]:
!nvcc ex01_cuda_basics.cu -o ex01
!./ex01

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
ex01_cuda_basics.cu(305): warning #177-D: variable "elapsed_ms" was declared but never referenced
          float elapsed_ms = 0.0f;
                ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"


  CUDA DIY Exercise 1: Basics & Memory
  GPU: Tesla T4  (SM 7.5)  VRAM: 15637 MB

[Section A] Reference:
  [A1-VectorAdd] N=1048576  CPU=5.9 ms  GPU=128.74 ms  Speedup=0.0x  [PASS]

[Section B] DIY Exercises:
  [B1-VectorScale] k=3.14  [FAIL] -- check your kernel
  [B2-SquaredDiff] [FAIL] -- check your kernel

  [B3-LaunchConfig] threads_per_block=256
           N    blocks    total_threads   covers_all?
  ---------------------------------------------------
           1         0                0        [FAIL]
         100         0                0        [FAIL]
         25

In [5]:
!nvcc ex02_memory_hierarchy.cu -o ex02
!./ex02

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
ex02_memory_hierarchy.cu(140): warning #177-D: variable "tile" was declared but never referenced
      __attribute__((shared)) float tile[256];
                                    ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

ex02_memory_hierarchy.cu(141): warning #177-D: variable "i" was declared but never referenced
      int i = blockIdx.x * blockDim.x + threadIdx.x;
          ^

ex02_memory_hierarchy.cu(198): warning #177-D: variable "i" was declared but never referenced
      int i = blockIdx.x * blockDim.x + tid;
          ^

ex02_memory_hierarchy.cu(277): warning #177-D: variable "REPS" was declared but never referenced
      int N = 1024, REPS = 5000;
                    ^


  CUDA DIY Exercise 2: Memory Hierarchy & Shared Mem
  GPU: Tesla T4  Shared mem/bloc

In [6]:
!nvcc ex03_ml_primitives.cu -o ex03
!./ex03

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
ex03_ml_primitives.cu(381): warning #177-D: variable "row" was declared but never referenced
      const float* row = logits + n * C;
                   ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"


  CUDA DIY Exercise 3: ML Primitives
  GPU: Tesla T4

[Section A] Reference:
  [A2-Softmax] Row sums = 1.0: [PASS]

[Section B] DIY Activations:
  [B1-Sigmoid] [FAIL] -- check expf formula
  [B2-Tanh] [FAIL]
  [B3-LeakyReLU] alpha=0.01  [FAIL]
  [B4-ReLUBackward] [FAIL]

[Section C] DIY Loss Functions:
  [C1-BCE-Loss] [FAIL] -- check clip + log formula
  [C2-CrossEntropy] N=512 C=10  [FAIL]

[Section D] Stretch — Adam Optimizer:
  [D1-Adam] 5 steps  [FAIL]

  All [PASS] = ready for Exercise 4!



In [8]:
!nvcc -arch=sm_75 ex04_cnn_layers.cu -o ex04 -lcublas
!./ex04

ex04_cnn_layers.cu(113): warning #550-D: variable "tA" was set but never used
      __attribute__((shared)) float tA[16][16];
                                    ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

ex04_cnn_layers.cu(114): warning #550-D: variable "tB" was set but never used
      __attribute__((shared)) float tB[16][16];
                                    ^


  CUDA DIY Exercise 4: Tiled GEMM & CNN Layers
  GPU: Tesla T4  Peak TFLOPS (FP32): ~130

[Section A] Reference: Naive MatMul:
  Naive 256x256@256x256  0.27 ms  124.6 GFLOPS

[Section B] DIY: Tiled GEMM:
  [B1-TiledMatMul] 512x512@512x512  0.01 ms  19373.2 GFLOPS  [FAIL]

  [B2-GemmBenchmark]
    Size     Naive(ms)     Tiled(ms)    cuBLAS(ms)  cuBLAS GFLOPS
  --------------------------------------------------------------
     128          0.03          0.00          0.00   4194304.0
     256          0.10          0.00          0.00  33554432.0
     512          0.73          0.00   

In [13]:
!mkdir -p data
%cd data

# Download from working source
!wget https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz
!wget https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz
!wget https://storage.googleapis.com/cvdf-datasets/mnist/t10k-images-idx3-ubyte.gz
!wget https://storage.googleapis.com/cvdf-datasets/mnist/t10k-labels-idx1-ubyte.gz

# Unzip
!gunzip *.gz

%cd ..

/content/data
--2026-04-27 17:49:46--  https://storage.googleapis.com/cvdf-datasets/mnist/train-images-idx3-ubyte.gz
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.70.207, 172.253.118.207, 74.125.200.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.70.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9912422 (9.5M) [application/x-gzip]
Saving to: ‘train-images-idx3-ubyte.gz’

train-images-idx3-u 100%[===================>]   9.45M  4.31MB/s    in 2.2s    

2026-04-27 17:49:50 (4.31 MB/s) - ‘train-images-idx3-ubyte.gz’ saved [9912422/9912422]

--2026-04-27 17:49:50--  https://storage.googleapis.com/cvdf-datasets/mnist/train-labels-idx1-ubyte.gz
Resolving storage.googleapis.com (storage.googleapis.com)... 172.217.70.207, 172.253.118.207, 74.125.200.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|172.217.70.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2888

In [14]:
!nvcc -arch=sm_75 ex05_mnist_cnn.cu -o ex05 -lcudnn -lcublas
!./ex05

ex05_mnist_cnn.cu(233): warning #177-D: variable "alpha" was declared but never referenced
      float alpha = 1.0f, beta = 0.0f;
            ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

ex05_mnist_cnn.cu(233): warning #177-D: variable "beta" was declared but never referenced
      float alpha = 1.0f, beta = 0.0f;
                          ^

ex05_mnist_cnn.cu(246): warning #177-D: variable "algo" was declared but never referenced
      cudnnConvolutionFwdAlgo_t algo =
                                ^

ex05_mnist_cnn.cu(259): warning #177-D: variable "ws_bytes" was declared but never referenced
      size_t ws_bytes = 0;
             ^

ex05_mnist_cnn.cu(300): warning #177-D: variable "alpha" was declared but never referenced
      float alpha = 1.0f, beta = 0.0f;
            ^

ex05_mnist_cnn.cu(300): warning #177-D: variable "beta" was declared but never referenced
      float alpha = 1.0f, beta = 0.0f;
                          ^

ex05_mnist_cnn